# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [ ]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [ ]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Attempt this in the code cell below: it is provided commented out, because both are sound and complete on `build_general_kb`'s output but neither scales to the full grid, and it is expected to hang or take an impractically long time. Uncomment a few lines at a time and give each at most about 30 seconds; use Kernel > Interrupt if it hasn't returned by then, and note what you observed.

Explain in your own words: what specifically makes `pl_resolution`'s cost grow out of control here, and separately, what makes `tt_entails`'s cost grow out of control? A simple complexity argument for each is the expected answer.

**Use your observations in Conceptual Question 2 below.**


In [17]:
# Attempt (a): try solving the puzzle using pl_resolution / tt_entails on the
# general KB. Left commented out because it is expected to take an
# impractically long time (pl_resolution) or be outright infeasible
# (tt_entails) on the full grid.
#
# Uncomment a few lines at a time and give each at most ~30 seconds; use
# Kernel > Interrupt if it hasn't returned by then.
#
general_kb = build_general_kb(n, box_h, box_w, givens)
r, c, v = 1, 1, givens.get((1, 1), 1)  # try one cell/value pair
query = atom('Is', r, c, v)
#
# print(pl_resolution(general_kb, query))    
print(tt_entails(associate('&', general_kb.clauses), query)) 

KeyboardInterrupt: 

### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.

**Resolution:** I made three attempts and manually interrupted execution after approximately 25 seconds, one minute, and two minutes, respectively. None of these attempts returned a boolean result before interruption.

In all three attempts, the `KeyboardInterrupt` traceback pointed to `new = new.union(set(resolvents))` inside `pl_resolution`, where resolution results are accumulated. This statement is executed repeatedly, so the identical interruption location does not establish that the algorithm was stuck in a single operation or an infinite loop.

These durations are observation limits, not completed solving times. The experiments show that the supplied resolution implementation did not answer this query within the observed time limits; they do not establish whether the query is entailed.

**Truth-table model checking:** I tested `tt_entails` on the general knowledge base for the 9×9 Sudoku puzzle using a single cell/value query. In three separate attempts, I manually interrupted execution after approximately 30 seconds, one minute, and two minutes. None returned a boolean result before interruption.

All three attempts produced a `KeyboardInterrupt`. The tracebacks showed deeply nested calls to `tt_check_all`, with execution interrupted while extending a partial truth assignment. These repeated stack frames indicate recursion depth, not the number of complete assignments checked.

The general encoding contains $9^3 = 729$ propositional symbols, giving $2^{729}$ possible truth assignments. The supplied implementation evaluates the knowledge base only after all symbols have been assigned, without pruning inconsistent partial assignments. It can terminate early upon finding an assignment where the knowledge base is true and the query is false, but otherwise the search space is extremely large.

These observations illustrate that theoretical completeness does not guarantee practical efficiency. The reported durations are observation limits, not completed solving times. The interruptions do not mean that the query is false or that the puzzle has no solution.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [ ]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [ ]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.
#
# import time
#
# t0 = time.time()
# solved = solve_full_grid_fc(n, box_h, box_w, givens)
# fc_time = time.time() - t0
# assert solved == puzzle['solution']
#
# definite_kb = build_definite_kb(n, box_h, box_w, givens)
#
# # Completeness: pl_bc_entails must find the correct value for every cell.
# for (r, c), v in puzzle['solution'].items():
#     assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True
#
# # Soundness: pl_bc_entails must not also confirm any incorrect value.
# for (r, c), v in puzzle['solution'].items():
#     for other_v in range(1, n + 1):
#         if other_v != v:
#             assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False
#
# t0 = time.time()
# solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
# bc_time = time.time() - t0
# assert solved_bc == puzzle['solution']
#
# print(f"solve_full_grid_fc: {fc_time:.2f}s")
# print(f"solve_full_grid_bc: {bc_time:.2f}s")

## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**
### 1(a) General KB Representation Strategy

In `build_general_kb`, I use the propositional symbol $Is_{r,c,v}$ to mean “the cell at row $r$, column $c$ contains value $v$.” The indices $r,c$ and value $v$ range from $1$ to $n$. Since the general representation supports logical negation directly, a separate `Not` symbol family is unnecessary.

I encode the Sudoku constraints as follows.

**1. At least one value per cell**

For every cell $(r,c)$, I add:

$$
Is_{r,c,1}\lor Is_{r,c,2}\lor\cdots\lor Is_{r,c,n}
$$

This requires at least one of the candidate-value propositions to be true.

**2. At most one value per cell**

For every cell and every pair of distinct values $v,w$, I add:

$$
\neg Is_{r,c,v}\lor\neg Is_{r,c,w}
$$

This states that the cell cannot contain both $v$ and $w$. The code considers only pairs with $v<w$ to avoid duplicate pairs. Together with the at-least-one constraint, this ensures that each cell contains exactly one value.

**3. Row uniqueness**

For each row $r$, value $v$, and pair of distinct columns $c_1,c_2$, I add:

$$
\neg Is_{r,c_1,v}\lor\neg Is_{r,c_2,v}
$$

This prevents two cells in the same row from both containing $v$.

**4. Column uniqueness**

For each column $c$, value $v$, and pair of distinct rows $r_1,r_2$, I add:

$$
\neg Is_{r_1,c,v}\lor\neg Is_{r_2,c,v}
$$

This prevents two cells in the same column from both containing $v$.

**5. Box uniqueness**

Using `box_h` and `box_w`, I collect the coordinates of all cells in each box. For every pair of distinct cells $(r_1,c_1)$, $(r_2,c_2)$ within that box and every value $v$, I add:

$$
\neg Is_{r_1,c_1,v}\lor\neg Is_{r_2,c_2,v}
$$

This prevents two cells in the same box from both containing $v$.

**6. Given values**

If `givens` specifies that cell $(r,c)$ contains $v$, I add the fact:

$$
Is_{r,c,v}
$$

All these constraints are in Conjunctive Normal Form (CNF): each clause is a disjunction of literals, and the knowledge base represents the conjunction of all its clauses. Each at-least-one constraint contains $n$ positive literals; each uniqueness constraint contains two negative literals; and each given is a unit clause. I add these constraints using `PropKB.tell()` and return the resulting `PropKB`.

### 1(b) Definite KB Representation Strategy

In `build_definite_kb`, I use `PropDefiniteKB`. A definite clause contains exactly one positive literal and can be written as:

$$
P_1\land P_2\land\cdots\land P_k\Rightarrow Q
$$

Here, the premises and conclusion are atomic propositions. Atomic facts are also permitted.

Consequently, the following constraint cannot be added directly:

$$
Is_{r,c,1}\lor Is_{r,c,2}\lor\cdots\lor Is_{r,c,n}
$$

It contains multiple positive literals. My implementation instead uses candidate-elimination rules and last-candidate rules.

**1. Introduce two families of atomic propositions**

- $Is_{r,c,v}$: cell $(r,c)$ contains value $v$.
- $Not_{r,c,v}$: candidate value $v$ has been eliminated from cell $(r,c)$.

`Not` is an independent propositional symbol, not a logical negation operator. The system does not automatically equate $Not_{r,c,v}$ with $\neg Is_{r,c,v}$; its role is established by the inference rules I add. This allows an elimination result to appear as the atomic conclusion of a definite rule.

**2. Eliminate other values from the same cell**

For every cell and all values $v\ne w$, I add:

$$
Is_{r,c,v}\Rightarrow Not_{r,c,w}
$$

Once a cell is known to contain $v$, its other candidate values are eliminated. Since implication is directional, the code considers all ordered pairs of distinct values, rather than only pairs with $v<w$.

**3. Eliminate the same value from other cells in the same row, column, or box**

For two distinct cells in the same row:

$$
Is_{r,c_1,v}\Rightarrow Not_{r,c_2,v}
\qquad(c_1\ne c_2)
$$

For two distinct cells in the same column:

$$
Is_{r_1,c,v}\Rightarrow Not_{r_2,c,v}
\qquad(r_1\ne r_2)
$$

For two distinct cells in the same box:

$$
Is_{r_1,c_1,v}\Rightarrow Not_{r_2,c_2,v}
$$

These rules state that once a cell is known to contain $v$, every other cell sharing its row, column, or box must exclude $v$. I generate rules in both directions for each pair of related cells.

**4. Determine new values using last-candidate rules**

For every cell and every candidate value $v$, I add:

$$
\left(
\bigwedge_{\substack{w=1\\w\ne v}}^{n}
Not_{r,c,w}
\right)
\Rightarrow Is_{r,c,v}
$$

If all other candidate values have been eliminated, the cell must contain $v$.

For example, in a $4\times4$ Sudoku:

$$
Not_{r,c,1}\land Not_{r,c,2}\land Not_{r,c,3}
\Rightarrow Is_{r,c,4}
$$

The premises must be connected by AND because every other candidate must be eliminated before the remaining value can be determined.

**5. Add given facts to initiate inference**

I add each value in `givens` as an $Is_{r,c,v}$ fact. Starting from these facts, the inference algorithm applies elimination rules to derive `Not` propositions, then applies last-candidate rules to derive new `Is` propositions, allowing further propagation.

This definite encoding represents valid elimination-based Sudoku reasoning rather than a fully equivalent rewriting of the general constraints. For example, deriving the remaining candidate after all alternatives have been eliminated is not equivalent to directly asserting that at least one candidate is true. Therefore, inference may stop when no further candidates can be eliminated, even if the puzzle has a unique solution. The completeness of forward or backward chaining for this Horn knowledge base does not imply that this limited set of Sudoku rules can solve every Sudoku puzzle.

### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**

I would not use either of the supplied implementations as the default algorithm for solving full 9×9 Sudoku puzzles. Soundness and completeness guarantee logically correct inference, but they do not guarantee an answer within practical time and memory limits.

### Truth-table model checking

With $m$ propositional symbols, truth-table checking may examine all $2^m$ truth assignments, evaluating the knowledge base and query for each assignment. Our Sudoku encoding contains $9^3 = 729$ symbols, giving $2^{729}$ possible assignments.

The supplied implementation checks the knowledge base only after constructing a complete assignment; it does not prune partial assignments that already violate Sudoku constraints. Although finding a counterexample allows early termination, an entailed query requires exhaustive checking in this implementation.

The algorithm explores assignments recursively rather than storing the entire truth table, so its memory use is not proportional to $2^m$. Nevertheless, its exponential running time makes it impractical at this scale.

### Resolution-refutation

Resolution avoids enumerating complete truth assignments, but it can generate a very large collection of intermediate clauses. With $k$ current clauses, the supplied implementation explicitly constructs $k(k-1)/2$ clause pairs in each iteration. This creates quadratic pair-list memory overhead and requires processing each pair. As new clauses are added, subsequent iterations become more expensive. The number of possible distinct clauses can grow exponentially with the number of propositional symbols.

An empty clause permits early termination, but the implementation first constructs the entire pair list for that iteration. Therefore, even a query with a short proof can incur substantial overhead.

### Experimental evidence and conclusion

For resolution, I interrupted separate attempts after approximately 25 seconds, one minute, and two minutes. For truth-table checking, I interrupted separate attempts after approximately 30 seconds, one minute, and two minutes. Neither algorithm returned a boolean answer before interruption, even for a single cell/value query.


### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**

### Pseudocode:
<pre>
<b>function</b> pl_bc_entails(kb, query) returns true or false
    inputs: kb: the knowledge base, a set of propositional definite clauses
            query: a propositional symbol

    <b>return</b> prove(query, empty set)

<b>function</b> prove(q, visiting) returns true or false
    inputs: q: the current query
            visiting: a set of queries on the current recursion path

    <b>if</b> q matches a fact in kb, <b>then return true</b>
    <b>if</b> q in visiting, <b>then return false</b>
    new_visiting = visiting ∪ {q}
    rules = all clauses in KB whose consequent matches q
    <b>if</b> there is no clause with a consequent that matches q, <b>then return</b> false

    <b>for each</b> clause c in kb where q is in c.CONCLUSION <b>do</b>
        count = number of symbols in c.PREMISE

        <b>for all</b> symbol p in c.PREMISE <b>do</b>
            <b>if</b> prove(p, new_visiting)
                <b>if</b> p not in kb, <b>then</b> add p to KB
                count = count - 1
            <b>else break</b>
        <b>if</b> count == 0, <b>then return </b>true

    <b>return</b> false
</pre>

### Design Challenges
The first challenge is to make sure that the backward chaining algorithm can terminate when there are cycles in the knowledge base. For example, if there are two rules `A => B` and `B => A`, the algorithm may recursively try to prove A from B and then B from A forever. To solve this problem, I use a `visiting` set to record the goals on the current recursion path. Before recursively proving a goal, the algorithm checks whether it is already in `visiting`. If it is, the current path contains a cycle, so the algorithm returns false for this path. I use `new_visiting = visiting ∪ {q}` so that each recursive branch has its own set of visited goals and other possible proof paths are not incorrectly blocked.

The second challenge is to correctly handle multiple premises in one rule. For a rule such as `A & B => C`, both A and B must be proved before C can be proved. Therefore, I use `count` to record the number of premises that still need to be proved. Every time a premise is successfully proved, `count` is decreased by one. Only when `count == 0` does the algorithm return true for the current goal.

The third challenge is that there may be multiple rules whose conclusions match the same goal. The algorithm must try these rules separately because only one of them needs to succeed to prove the goal. Therefore, if one rule fails because one of its premises cannot be proved, the algorithm continues to try the next candidate rule. It returns true as soon as one complete rule succeeds, and returns false only after all candidate rules have failed.

### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**



### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**Your answer:**



## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

Paste your deployed Streamlit app URL here:

`https://...`
